In [1]:
import carla, random, pygame, numpy as np

pygame 2.6.1 (SDL 2.28.4, Python 3.10.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
# Core elements in carla are: actors, blueprints, worlds, and clients
client = carla.Client('localhost', 2000)

# Set the map to Town06. Silence output
try:
    client.load_world('Town06')
except RuntimeError:
    pass  # Ignore timeout or connection errors

In [3]:
# Get the world object
world = client.get_world()

# settings = world.get_settings()
# settings.synchronous_mode = True # Enables synchronous mode. THIS BREAKS THE SIM
# settings.fixed_delta_seconds = 0.01 # Set a variable time-step
# world.apply_settings(settings)

In [12]:
# Retrieve the spectator object
spectator = world.get_spectator()

# Update the spectator's location and rotation to the center lane of our scenario
# default_location = carla.Location(x=17.092312, y=244.485397, z=0.500000)
# default_rotation = carla.Rotation(pitch=360.000000, yaw=1.780838, roll=0.000000)

# Top-down view of our scenario
default_location = carla.Location(x=116, y=244.485397, z=100.000000) # At a height of z=100 meters, track length is about 200 meters
default_rotation = carla.Rotation(pitch=270.000000, yaw=270, roll=0.000000)

spectator.set_transform(carla.Transform(default_location, default_rotation))

In [ ]:
# Spawn our ego and NPC

ego_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_2020')
ego_spawn_location = carla.Location(x=21, y=244.485397, z=0.500000)
ego_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
ego_spawn_point = carla.Transform(ego_spawn_location, ego_spawn_rotation)
ego_blueprint.set_attribute('role_name', 'hero') # Set the role name of the ego vehicle

npc_blueprint = world.get_blueprint_library().find('vehicle.dodge.charger_police_2020')
npc_spawn_location = carla.Location(x=21, y=(244.485397 - 3.5), z=0.500000)
npc_spawn_rotation = carla.Rotation(pitch=0.000000, yaw=0, roll=0.000000)
npc_spawn_point = carla.Transform(npc_spawn_location, npc_spawn_rotation)

ego_vehicle = world.spawn_actor(ego_blueprint, ego_spawn_point)
npc_vehicle = world.spawn_actor(npc_blueprint, npc_spawn_point)

# Height and width of window
W, H = 758, 396

cam_bp = world.get_blueprint_library().find("sensor.camera.rgb")
cam_bp.set_attribute("image_size_x", str(W))
cam_bp.set_attribute("image_size_y", str(H))
cam_bp.set_attribute("fov", "135")  # Set FOV in degrees (default is 90, higher = wider view)

# Driver's point of view: positioned at driver's head, looking forward through windshield
camera = world.spawn_actor(cam_bp, carla.Transform(
    carla.Location(x=0.2, y=-0.36, z=1.2),  # Driver's head position (x=0.5 for front/back, y=-0.36 for left/right, z=1.3 for head height)
    carla.Rotation(pitch=-10, yaw=0, roll=0)  # Slight downward pitch for natural driving gaze
), attach_to=ego_vehicle)

# Import the display handler
from carla_display import CarlaDisplay

# Create display handler (runs pygame in background thread)
display = CarlaDisplay(width=W, height=H)
display.start()

# Set up camera callback to send images to display
camera.listen(display.on_image)

# Enable NPC autopilot
# npc_vehicle.set_autopilot(True)
# ego_vehicle.set_autopilot(True)

# Now you can continue executing cells - the display runs in the background!
# To stop the display, run: display.stop()

Pygame display started in background thread


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


In [ ]:
# Auto-respawn monitoring system
import threading
import time

class AutoRespawnMonitor:
    """Monitors vehicles and respawns them when conditions are met."""
    
    def __init__(self, world, ego_spawn_point, npc_spawn_point, ego_blueprint, npc_blueprint, 
                 camera_blueprint, camera_transform, display, spawn_x_threshold=200, time_threshold=30):
        self.world = world
        self.ego_spawn_point = ego_spawn_point
        self.npc_spawn_point = npc_spawn_point
        self.ego_blueprint = ego_blueprint
        self.npc_blueprint = npc_blueprint
        self.camera_blueprint = camera_blueprint
        self.camera_transform = camera_transform
        self.display = display
        
        self.spawn_x = ego_spawn_point.location.x
        self.x_threshold = self.spawn_x + spawn_x_threshold
        self.time_threshold = time_threshold
        
        self.ego_vehicle = None
        self.npc_vehicle = None
        self.camera = None
        
        self.running = False
        self.monitor_thread = None
        
    def respawn_vehicles(self):
        """Destroy and respawn both vehicles and camera."""
        print("Respawn triggered!")
        
        # Destroy existing actors
        if self.camera:
            self.camera.stop()
            self.camera.destroy()
        if self.ego_vehicle:
            self.ego_vehicle.destroy()
        if self.npc_vehicle:
            self.npc_vehicle.destroy()
        
        # Spawn new vehicles
        self.ego_vehicle = self.world.spawn_actor(self.ego_blueprint, self.ego_spawn_point)
        self.npc_vehicle = self.world.spawn_actor(self.npc_blueprint, self.npc_spawn_point)
        
        # Spawn and attach camera
        self.camera = self.world.spawn_actor(
            self.camera_blueprint, 
            self.camera_transform, 
            attach_to=self.ego_vehicle
        )
        self.camera.listen(self.display.on_image)
        
        print(f"Vehicles respawned at x={self.spawn_x}")
    
    def _monitor_loop(self):
        """Main monitoring loop running in background thread."""
        start_time = time.time()
        
        while self.running:
            if self.ego_vehicle is None or self.npc_vehicle is None:
                time.sleep(0.1)
                continue
            
            # Check time condition
            elapsed_time = time.time() - start_time
            
            # Check position condition
            try:
                ego_location = self.ego_vehicle.get_location()
                npc_location = self.npc_vehicle.get_location()
                ego_x = ego_location.x
                npc_x = npc_location.x
                
                # Check if either vehicle has traveled x += 200 from spawn
                position_triggered = (ego_x >= self.x_threshold) or (npc_x >= self.x_threshold)
                
                # Check if 30 seconds have passed
                time_triggered = elapsed_time >= self.time_threshold
                
                if position_triggered or time_triggered:
                    trigger_reason = []
                    if position_triggered:
                        trigger_reason.append(f"Position (ego_x={ego_x:.2f}, npc_x={npc_x:.2f} >= {self.x_threshold})")
                    if time_triggered:
                        trigger_reason.append(f"Time ({elapsed_time:.2f}s >= {self.time_threshold}s)")
                    print(f"Respawn condition met: {', '.join(trigger_reason)}")
                    
                    self.respawn_vehicles()
                    start_time = time.time()  # Reset timer after respawn
                    
            except Exception as e:
                # Vehicle might have been destroyed externally
                print(f"Error monitoring vehicles: {e}")
                time.sleep(0.1)
                continue
            
            time.sleep(0.1)  # Check every 100ms
    
    def start(self, ego_vehicle, npc_vehicle, camera):
        """Start monitoring with initial vehicles."""
        self.ego_vehicle = ego_vehicle
        self.npc_vehicle = npc_vehicle
        self.camera = camera
        
        if not self.running:
            self.running = True
            self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
            self.monitor_thread.start()
            print("Auto-respawn monitor started")
    
    def stop(self):
        """Stop monitoring."""
        if self.running:
            self.running = False
            if self.monitor_thread:
                self.monitor_thread.join(timeout=2.0)
            print("Auto-respawn monitor stopped")

# Create monitor instance
monitor = AutoRespawnMonitor(
    world=world,
    ego_spawn_point=ego_spawn_point,
    npc_spawn_point=npc_spawn_point,
    ego_blueprint=ego_blueprint,
    npc_blueprint=npc_blueprint,
    camera_blueprint=cam_bp,
    camera_transform=carla.Transform(
        carla.Location(x=0.2, y=-0.36, z=1.2),
        carla.Rotation(pitch=-10, yaw=0, roll=0)
    ),
    display=display,
    spawn_x_threshold=200,  # x += 200 from spawn
    time_threshold=30  # 30 seconds
)

# Start monitoring
monitor.start(ego_vehicle, npc_vehicle, camera)

# To stop monitoring later, run: monitor.stop()


In [10]:
# To stop the display and camera:
display.stop()
camera.stop()

# Get the list of actors
actors = world.get_actors()

# Delete all vehicles
for actor in actors:
    if actor.type_id.startswith('vehicle') or actor.type_id.startswith('sensor'):
        # print(actor.type_id)
        actor.destroy()


Pygame display stopped
